In [111]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [112]:
#데이터 로드
data = pd.read_csv('Sleep Health and Lifestyle Dataset.csv')

In [113]:
#데이터 확인
data.info()
data.columns

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 39 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Patient_ID                     30000 non-null  int64  
 1   Age                            30000 non-null  int64  
 2   Gender                         30000 non-null  str    
 3   Occupation                     30000 non-null  str    
 4   Marital_Status                 30000 non-null  str    
 5   Physical_Activity_Minutes      30000 non-null  float64
 6   Screen_Time_Hours              30000 non-null  float64
 7   Daily_Steps                    30000 non-null  int64  
 8   Water_Intake_Liters            30000 non-null  float64
 9   Caffeine_Intake_mg             30000 non-null  int64  
 10  Smoking_Status                 30000 non-null  str    
 11  Alcohol_Consumption            30000 non-null  str    
 12  Bedtime_Consistency            30000 non-null  int64  
 1

Index(['Patient_ID', 'Age', 'Gender', 'Occupation', 'Marital_Status',
       'Physical_Activity_Minutes', 'Screen_Time_Hours', 'Daily_Steps',
       'Water_Intake_Liters', 'Caffeine_Intake_mg', 'Smoking_Status',
       'Alcohol_Consumption', 'Bedtime_Consistency', 'Sleep_Duration',
       'Sleep_Quality', 'Sleep_Efficiency', 'Sleep_Latency',
       'Wake_After_Sleep_Onset', 'Night_Awakenings', 'Daytime_Sleepiness',
       'Weekend_Sleep_Variation', 'BMI', 'Heart_Rate', 'Systolic_BP',
       'Diastolic_BP', 'SpO2', 'HRV', 'Stress_Level', 'Anxiety_Level',
       'Depression_Score', 'Workload_Score', 'Snoring_Frequency',
       'Respiratory_Disturbance_Index', 'Leg_Movement_Index',
       'Sleep_Risk_Index', 'Lifestyle_Risk_Index', 'Physiological_Risk_Index',
       'Mental_Health_Risk_Index', 'Sleep_Disorder'],
      dtype='str')

| 컬럼                              | 설명                                                                     |
| ------------------------------- | ---------------------------------------------------------------------- |
| `Patient_ID`                    | 각 사용자를 구분하기 위한 고유 번호. 예측에는 의미가 없으므로 보통 제거합니다.                          |
| `Age`                           | 나이. 데이터에서는 18~80세입니다.                                                  |
| `Gender`                        | 성별. `Male`, `Female`로 구성되어 있습니다.                                       |
| `Occupation`                    | 직업. Doctor, Engineer, Student, Teacher 등 직업군입니다.                       |
| `Marital_Status`                | 결혼 상태. `Married`, `Single`, `Divorced`입니다.                             |
| `Physical_Activity_Minutes`     | 하루 신체 활동 시간(분). 운동이나 활동량을 나타내는 변수입니다.                                  |
| `Screen_Time_Hours`             | 하루 스마트폰·PC·TV 등의 화면 사용 시간(시간).                                         |
| `Daily_Steps`                   | 하루 걸음 수. 활동량 지표입니다.                                                    |
| `Water_Intake_Liters`           | 하루 물 섭취량(L).                                                           |
| `Caffeine_Intake_mg`            | 하루 카페인 섭취량(mg). 커피, 에너지음료 등의 카페인 양을 의미합니다.                             |
| `Smoking_Status`                | 흡연 여부. `Yes`, `No`입니다.                                                 |
| `Alcohol_Consumption`           | 음주 여부. `Yes`, `No`입니다.                                                 |
| `Bedtime_Consistency`           | 취침 시간이 얼마나 규칙적인지를 나타내는 점수. 이 데이터에서는 3~9입니다. 높을수록 규칙적인 것으로 해석할 수 있습니다.  |
| `Sleep_Duration`                | 하루 수면 시간(시간). 예: `6.5` = 6시간 30분.                                      |
| `Sleep_Quality`                 | 주관적 수면의 질 점수. 데이터에서는 1~10입니다. 높을수록 수면의 질이 좋은 것으로 보입니다.                 |
| `Sleep_Efficiency`              | 침대에 누워 있던 시간 중 실제로 잠든 시간의 비율. 데이터에서는 약 57~99입니다. 일반적으로 `%`로 해석합니다.     |
| `Sleep_Latency`                 | 잠자리에 든 후 실제로 잠드는 데 걸린 시간. 보통 분 단위로 사용합니다.                              |
| `Wake_After_Sleep_Onset`        | 한번 잠든 이후 깨어 있었던 총 시간. 흔히 WASO라고 부르며 이 데이터에서는 0~60입니다.                  |
| `Night_Awakenings`              | 밤에 잠에서 깬 횟수.                                                           |
| `Daytime_Sleepiness`            | 낮 동안 느끼는 졸림 정도. 데이터에서는 1~10입니다.                                        |
| `Weekend_Sleep_Variation`       | 평일과 주말 사이 수면 시간 차이를 나타내는 값. 주말에 얼마나 수면 패턴이 달라지는지 보여줍니다.                |
| `BMI`                           | 체질량지수. 체중과 키를 이용한 비만도 지표입니다.                                           |
| `Heart_Rate`                    | 심박수. 일반적으로 분당 심장 박동수(bpm)를 의미합니다.                                      |
| `Systolic_BP`                   | 수축기 혈압. 혈압의 위쪽 숫자입니다. 예: `120/80`에서 120.                               |
| `Diastolic_BP`                  | 이완기 혈압. 혈압의 아래쪽 숫자입니다. 예: `120/80`에서 80.                               |
| `SpO2`                          | 혈중 산소포화도(%). 수면무호흡증과 관련해 볼 수 있는 지표입니다.                                 |
| `HRV`                           | Heart Rate Variability, 심박변이도. 심장 박동 간격의 변화를 나타내는 지표입니다.               |
| `Stress_Level`                  | 스트레스 수준. 이 데이터에서는 4~9입니다.                                              |
| `Anxiety_Level`                 | 불안 수준. 1~10 범위입니다.                                                     |
| `Depression_Score`              | 우울 관련 점수. 데이터에서는 0~10입니다.                                              |
| `Workload_Score`                | 업무 또는 학업 부담 정도를 나타내는 점수. 1~10입니다.                                      |
| `Snoring_Frequency`             | 코골이 빈도를 나타내는 점수. 이 데이터에서는 2~8입니다.                                      |
| `Respiratory_Disturbance_Index` | 수면 중 호흡 장애 정도를 나타내는 지표. 호흡 이상이 얼마나 자주 발생하는지를 나타내는 값으로 볼 수 있습니다.        |
| `Leg_Movement_Index`            | 수면 중 다리 움직임 정도를 나타내는 지표. 하지불안증후군 등의 예측과 관련될 수 있습니다.                    |
| `Sleep_Risk_Index`              | 수면 관련 여러 변수로 만들어진 것으로 보이는 종합 수면 위험 점수. 정확한 계산식은 CSV에는 없습니다.            |
| `Lifestyle_Risk_Index`          | 활동량, 카페인, 화면 사용 등 생활습관 변수들을 조합한 것으로 보이는 위험 점수. 정확한 산식은 파일에 없습니다.       |
| `Physiological_Risk_Index`      | 혈압, SpO2, BMI, 호흡 관련 값 등 생리적 상태를 종합한 것으로 보이는 위험 지수. 정확한 계산식은 파일에 없습니다. |
| `Mental_Health_Risk_Index`      | 스트레스, 불안, 우울 등의 정신건강 관련 정보를 종합한 것으로 보이는 위험 지수. 정확한 계산식은 파일에 없습니다.      |
| `Sleep_Disorder`                | 수면 장애 컬럼                                          |


In [114]:
#결측치 확인
data.isnull().sum()

Patient_ID                       0
Age                              0
Gender                           0
Occupation                       0
Marital_Status                   0
Physical_Activity_Minutes        0
Screen_Time_Hours                0
Daily_Steps                      0
Water_Intake_Liters              0
Caffeine_Intake_mg               0
Smoking_Status                   0
Alcohol_Consumption              0
Bedtime_Consistency              0
Sleep_Duration                   0
Sleep_Quality                    0
Sleep_Efficiency                 0
Sleep_Latency                    0
Wake_After_Sleep_Onset           0
Night_Awakenings                 0
Daytime_Sleepiness               0
Weekend_Sleep_Variation          0
BMI                              0
Heart_Rate                       0
Systolic_BP                      0
Diastolic_BP                     0
SpO2                             0
HRV                              0
Stress_Level                     0
Anxiety_Level       

In [115]:
#중복치 확인
data.duplicated().sum()

np.int64(0)

In [116]:
data.columns

Index(['Patient_ID', 'Age', 'Gender', 'Occupation', 'Marital_Status',
       'Physical_Activity_Minutes', 'Screen_Time_Hours', 'Daily_Steps',
       'Water_Intake_Liters', 'Caffeine_Intake_mg', 'Smoking_Status',
       'Alcohol_Consumption', 'Bedtime_Consistency', 'Sleep_Duration',
       'Sleep_Quality', 'Sleep_Efficiency', 'Sleep_Latency',
       'Wake_After_Sleep_Onset', 'Night_Awakenings', 'Daytime_Sleepiness',
       'Weekend_Sleep_Variation', 'BMI', 'Heart_Rate', 'Systolic_BP',
       'Diastolic_BP', 'SpO2', 'HRV', 'Stress_Level', 'Anxiety_Level',
       'Depression_Score', 'Workload_Score', 'Snoring_Frequency',
       'Respiratory_Disturbance_Index', 'Leg_Movement_Index',
       'Sleep_Risk_Index', 'Lifestyle_Risk_Index', 'Physiological_Risk_Index',
       'Mental_Health_Risk_Index', 'Sleep_Disorder'],
      dtype='str')

In [117]:
# 스트레스 수준을 범주형으로 변경
data["Stress_Level"] = data["Stress_Level"].map({
    4: "Low",
    7: "Medium",
    9: "High"
})

In [118]:
#데이터 분리 

features = [ #사용할 데이터
    "Age",                          # 나이
    "Gender",                       # 성별
    "BMI",                          # 체질량지수
    "Sleep_Duration",               # 평균 수면 시간
    "Caffeine_Intake_mg",           # 하루 카페인 섭취량
    "Stress_Level",                 # 스트레스 수준
    "Alcohol_Consumption",          # 음주 여부
    "Physical_Activity_Minutes",    # 하루 운동 시간
    "Smoking_Status"                # 흡연 여부
]

X = data[features]                        
y = data["Lifestyle_Risk_Index"]          # 생활습관 위험 지수 예측

In [119]:
X.shape, y.shape

((30000, 9), (30000,))

In [120]:
# 숫자형 칼럼, 범주형 칼럼
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "str"]).columns.tolist()

In [121]:
# 숫자형 
numeric_features

['Age',
 'BMI',
 'Sleep_Duration',
 'Caffeine_Intake_mg',
 'Physical_Activity_Minutes']

In [122]:
#범주형
categorical_features

['Gender', 'Stress_Level', 'Alcohol_Consumption', 'Smoking_Status']

In [123]:
# 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [124]:
# numeric data 숫자형 데이터 표준화
scaler_standard = StandardScaler()

X_train[numeric_features] = scaler_standard.fit_transform(
    X_train[numeric_features]
)

X_test[numeric_features] = scaler_standard.transform(
    X_test[numeric_features]
)

In [125]:
#범주형 데이터 One-Hot Encoding
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_encoded = encoder.transform(
    X_test[categorical_features]
)

In [126]:
encoder.get_feature_names_out(categorical_features)

array(['Gender_Female', 'Gender_Male', 'Stress_Level_High',
       'Stress_Level_Low', 'Stress_Level_Medium',
       'Alcohol_Consumption_No', 'Alcohol_Consumption_Yes',
       'Smoking_Status_No', 'Smoking_Status_Yes'], dtype=object)

In [127]:
#데이터 합치기
X_train_final = np.hstack([
    X_train[numeric_features].values,
    X_train_encoded
])

X_test_final = np.hstack([
    X_test[numeric_features].values,
    X_test_encoded
])

In [128]:
X_train_final.shape, X_test_final.shape

((24000, 14), (6000, 14))

In [129]:
import joblib

In [130]:
preprocessing_data = {
    "X_train": X_train_final,
    "X_test": X_test_final,
    "y_train": y_train,
    "y_test": y_test,
    "scaler": scaler_standard,
    "encoder": encoder,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}

In [131]:
joblib.dump(preprocessing_data, "sleep_preprocessing_yt_요구사항ver.pkl")
print("전처리 데이터 저장 완료")

전처리 데이터 저장 완료
